In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import numpy as np 
from typing import Final 
import h5py
import matplotlib.pyplot as plt
import time 
import xarray as xr
from utils import plot_bias
import gc 


In [4]:
data_path = "data/"
lw_file = data_path + 'rrtmgp-data-lw-g128-210809_NN_GCM_NWP.nc'

lw_file

In [5]:
lw_ds.variables

Frozen({'nn_dimsize': <xarray.Variable (nn_layers: 3)> Size: 12B
[3 values with dtype=int32]
Attributes:
    long_name:  Dimension of each layer, not including the input layer, 'nn_activation': <xarray.Variable (nn_layers: 3)> Size: 96B
[3 values with dtype=<U8], 'nn_inputs': <xarray.Variable (nn_dim_input: 18)> Size: 504B
[18 values with dtype=<U7]
Attributes:
    long_name:  Specifies the inputs in their correct order, 'nn_input_coeffs_max': <xarray.Variable (nn_dim_input: 18)> Size: 72B
[18 values with dtype=float32]
Attributes:
    long_name:  xmax, see global attribute input_scaling_info, 'nn_input_coeffs_min': <xarray.Variable (nn_dim_input: 18)> Size: 72B
[18 values with dtype=float32]
Attributes:
    long_name:  xmin, see global attribute input_scaling_info, 'nn_activation_char': <xarray.Variable (nn_layers: 3)> Size: 96B
[3 values with dtype=|S32], 'nn_inputs_char': <xarray.Variable (nn_dim_input: 18)> Size: 576B
[18 values with dtype=|S32], 'nn_weights_1': <xarray.Variable (n

In [23]:
nn_lw_w1 = lw_ds['nn_weights_1'][:].values
nn_lw_w2 = lw_ds['nn_weights_2'][:].values
nn_lw_w3 = lw_ds['nn_weights_3'][:].values

nn_lw_b1 = lw_ds['nn_bias_1'][:].values
nn_lw_b2 = lw_ds['nn_bias_2'][:].values
nn_lw_b3 = lw_ds['nn_bias_3'][:].values

ynorm_lw_mean = lw_ds['nn_output_coeffs_mean'][:].values 
ynorm_lw_std =  lw_ds['nn_output_coeffs_std'][:].values 

xnorm_lw_max = lw_ds['nn_input_coeffs_max'][:].values 
xnorm_lw_min =  lw_ds['nn_input_coeffs_min'][:].values 

activ_str = lw_ds.nn_activation
input_str = lw_ds.nn_inputs

ynorm_str = lw_ds.output_scaling_info
xnorm_str = lw_ds.input_scaling_info

print("activation funcs:", activ_str[0].values, activ_str[1].values)
print("inputs:", input_str.values)
print(xnorm_str)
print(ynorm_str)

activation funcs: softsign softsign
inputs: ['tlay' 'play' 'h2o' 'o3' 'co2' 'ch4' 'n2o' 'cfc11' 'cfc12' 'co' 'ccl4'
 'cfc22' 'hfc143a' 'hfc125' 'hfc23' 'hfc32' 'hfc134a' 'cf4']
To get the required NN inputs, do the following: x(i) = log(x(i)) for i=pressure; x(i) = x(i)**(1/4) for i=H2O and O3; x(i) = (x(i) - xmin(i)) / (xmax(i) - xmin(i)) for all inputs
Model predicts scaled cross-sections. Given the raw NN output y, do the following to obtain optical depth: y(igpt,j) = ystd(igpt)*y(igpt,j) + ymean(igpt); y(igpt,j) = y(igpt,j)**8; y(igpt,j) = y(igpt,j) * layer_dry_air_molecules(j)


In [24]:
'cfc11' in input_str.values

True

In [7]:
print(xnorm_lw_max.shape[0])
print(xnorm_lw_max)
print(xnorm_lw_min)


18
[3.4000000e+02 1.1600000e+01 5.0775301e-01 6.3168339e-02 2.7999999e-03
 4.2000001e-06 5.8135214e-07 2.0000002e-09 5.9999999e-10 2.4000001e-06
 1.0316801e-10 2.3845328e-10 7.7914392e-10 9.8880004e-10 3.1067642e-11
 1.3642075e-11 4.2330001e-10 1.6702625e-10]
[1.60e+02 5.15e-03 1.01e-02 4.36e-03 1.41e-04 2.00e-09 0.00e+00 0.00e+00
 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00
 0.00e+00 0.00e+00]


In [14]:
xnorm_lw_min

array([1.60e+02, 5.15e-03, 1.01e-02, 4.36e-03, 1.41e-04, 2.00e-09,
       0.00e+00, 0.00e+00, 0.00e+00, 0.00e+00, 0.00e+00, 0.00e+00,
       0.00e+00, 0.00e+00, 0.00e+00, 0.00e+00, 0.00e+00, 0.00e+00],
      dtype=float32)

In [15]:
xnorm_lw_max

array([3.4000000e+02, 1.1600000e+01, 5.0775301e-01, 6.3168339e-02,
       2.7999999e-03, 4.2000001e-06, 5.8135214e-07, 2.0000002e-09,
       5.9999999e-10, 2.4000001e-06, 1.0316801e-10, 2.3845328e-10,
       7.7914392e-10, 9.8880004e-10, 3.1067642e-11, 1.3642075e-11,
       4.2330001e-10, 1.6702625e-10], dtype=float32)

In [6]:
ynorm_lw_std.shape

(256,)

In [79]:
nn_lw_w1.T.shape

(64, 18)

In [83]:
nn_lw_w1.T[0,:]

array([-1.77298829e-01,  1.18822789e+00,  1.03324927e-01,  7.42780328e-01,
       -2.79399436e-02, -8.34331557e-04, -1.64817926e-03,  8.75676997e-05,
       -5.14334184e-04, -1.32141765e-02,  2.94773548e-04,  3.37788428e-04,
       -1.55394850e-03, -2.25536735e-03, -1.07012027e-04, -3.35702352e-04,
       -1.32495514e-03, -4.94883920e-04], dtype=float32)

In [102]:
nn_lw_b1

array([-1.2667133 ,  1.2655754 ,  1.147521  ,  0.42550185,  0.3180539 ,
        2.2710836 ,  0.2567812 ,  0.8040583 , -0.5311072 , -1.356366  ,
        2.359002  ,  1.2974443 , -1.3865901 ,  0.0313884 , -0.5104355 ,
        0.38556832, -0.723964  ,  0.04967897, -0.6618776 ,  0.43231046,
        0.4456944 , -1.0574782 , -0.6047107 , -0.3398347 ,  1.1693499 ,
       -0.75573343, -0.40134555,  0.96278495,  1.4382306 , -0.5862757 ,
        0.4851271 ,  1.8005056 , -0.01952639,  0.21519932,  0.60845506,
       -0.21330771, -0.03355258,  1.5538994 , -0.17299879,  0.29405418,
       -0.13240941,  1.3663362 , -2.4562368 ,  0.82613206, -0.15454999,
       -0.14608705, -3.1358702 ,  0.88309014,  0.84012514,  1.7643992 ,
        2.1951878 ,  0.06374736,  1.4564338 , -1.105194  , -0.22207943,
        1.4903905 , -0.3267442 , -1.6242245 ,  0.7763965 ,  0.22849482,
       -1.2038794 , -0.09898248, -0.2625809 , -0.78977346], dtype=float32)

In [7]:
class gasopt_mlp_lw(nn.Module):
    additional_mlp: Final[bool]
    def __init__(self, ng,
                xmin, xmax, ymean, ystd,
                nn_lw_w1, nn_lw_w2, nn_lw_w3,
                nn_lw_b1, nn_lw_b2, nn_lw_b3):
        super(gasopt_mlp_lw, self).__init__()
        xmin  = torch.from_numpy(xmin)
        xmax  = torch.from_numpy(xmax)
        ymean = torch.from_numpy(ymean)
        ystd  = torch.from_numpy(ystd)
        
        self.register_buffer('xmin', xmin)
        self.register_buffer('xmax', xmax)
        self.register_buffer('ymean', ymean)
        self.register_buffer('ystd', ystd)
        self.nx = xmin.shape[0]
        self.ny = ymean.shape[0]
        self.ng0 = self.ny//2
        self.ng = ng
        self.nh = nn_lw_w1.shape[1]
        self.softsign =  nn.Softsign()
        
        self.mlp1 = nn.Linear(self.nx, self.nh)
        self.mlp2 = nn.Linear(self.nh, self.nh)
        self.mlp3 = nn.Linear(self.nh, self.ny)
        if self.ng != self.ng0:
            self.additional_mlp = True
            self.mlp4 = nn.Linear(self.ny, self.ng*2)
        else:
            self.additional_mlp = False 
        print("gasopt_mlp_lw number of original g-points: {}, new: {}, hidden neurons: {}, add MLP: {}".format(self.ng0, 
                                                                                                    self.ng, self.nh, self.additional_mlp)) 

        self.mlp1.weight = torch.nn.Parameter(torch.from_numpy(nn_lw_w1.T))
        self.mlp2.weight = torch.nn.Parameter(torch.from_numpy(nn_lw_w2.T))
        self.mlp3.weight = torch.nn.Parameter(torch.from_numpy(nn_lw_w3.T))
        self.mlp1.bias = torch.nn.Parameter(torch.from_numpy(nn_lw_b1.T))
        self.mlp2.bias = torch.nn.Parameter(torch.from_numpy(nn_lw_b2.T))
        self.mlp3.bias = torch.nn.Parameter(torch.from_numpy(nn_lw_b3.T))
        self.mlp1.weight.requires_grad = False; self.mlp1.bias.requires_grad = False; 
        self.mlp2.weight.requires_grad = False; self.mlp2.bias.requires_grad = False
        self.mlp3.weight.requires_grad = False; self.mlp3.bias.requires_grad = False

    def forward(self, x, col_dry=None):
        x = self.mlp1(x)
        x = self.softsign(x)
        x = self.mlp2(x)
        x = self.softsign(x)
        x = self.mlp3(x)
        if self.additional_mlp:
            x = self.mlp4(x)
        tau, pfrac = x.chunk(2,-1)

        if col_dry is not None:
            tau = col_dry * torch.pow(self.ystd*tau + self.ymean,8)
            pfrac = torch.square(pfrac)
        #    ! Postprocess absorption output: reverse standard scaling and square root scaling
        #    tau(igpt,ilay,icol) = (ystd(igpt) * outp_both(igpt,ilay,icol) + ymeans(igpt))**8
        #    ! Optical depth from cross-sections
        #    tau(igpt,ilay,icol) = tau(igpt,ilay,icol)*col_dry_wk(ilay,icol)
        #    ! Planck fraction: reverse square root scaling
        #    pfrac(igpt,ilay,icol) = outp_both(igpt+ngpt,ilay,icol)*outp_both(igpt+ngpt,ilay,icol)
        #y(igpt,j) = ystd(igpt)*y(igpt,j) + ymean(igpt); y(igpt,j) = y(igpt,j)**8; y(igpt,j) = y(igpt,j) * layer_dry_air_molecules(j)
    
        return tau, pfrac

class gasopt_mlp_sw(nn.Module):
    additional_mlp: Final[bool]
    def __init__(self, ng,
                xmin, xmax, ymean, ystd,
                nn_sw_w1, nn_sw_w2, nn_sw_w3,
                nn_sw_b1, nn_sw_b2, nn_sw_b3):
        super(gasopt_mlp_lw, self).__init__()
        xmin  = torch.from_numpy(xmin)
        xmax  = torch.from_numpy(xmax)
        ymean = torch.from_numpy(ymean)
        ystd  = torch.from_numpy(ystd)
        
        self.register_buffer('xmin', xmin)
        self.register_buffer('xmax', xmax)
        self.register_buffer('ymean', ymean)
        self.register_buffer('ystd', ystd)
        self.nx = xmin.shape[0]
        self.ny = ymean.shape[0]
        self.ng0 = self.ny//2
        self.ng = ng
        self.nh = nn_lw_w1.shape[1]
        self.softsign =  nn.Softsign()
        
        self.mlp1 = nn.Linear(self.nx, self.nh)
        self.mlp2 = nn.Linear(self.nh, self.nh)
        self.mlp3 = nn.Linear(self.nh, self.ny)
        if self.ng != self.ng0:
            self.additional_mlp = True
            self.mlp4 = nn.Linear(self.ny, self.ng*2)
        else:
            self.additional_mlp = False 
        print("gasopt_mlp_lw number of original g-points: {}, new: {}, hidden neurons: {}, add MLP: {}".format(self.ng0, 
                                                                                                    self.ng, self.nh, self.additional_mlp)) 

        self.mlp1.weight = torch.nn.Parameter(torch.from_numpy(nn_lw_w1.T))
        self.mlp2.weight = torch.nn.Parameter(torch.from_numpy(nn_lw_w2.T))
        self.mlp3.weight = torch.nn.Parameter(torch.from_numpy(nn_lw_w3.T))
        self.mlp1.bias = torch.nn.Parameter(torch.from_numpy(nn_lw_b1.T))
        self.mlp2.bias = torch.nn.Parameter(torch.from_numpy(nn_lw_b2.T))
        self.mlp3.bias = torch.nn.Parameter(torch.from_numpy(nn_lw_b3.T))
        self.mlp1.weight.requires_grad = False; self.mlp1.bias.requires_grad = False; 
        self.mlp2.weight.requires_grad = False; self.mlp2.bias.requires_grad = False
        self.mlp3.weight.requires_grad = False; self.mlp3.bias.requires_grad = False

    def forward(self, x, col_dry=None):
        x = self.mlp1(x)
        x = self.softsign(x)
        x = self.mlp2(x)
        x = self.softsign(x)
        x = self.mlp3(x)
        if self.additional_mlp:
            x = self.mlp4(x)
        tau, pfrac = x.chunk(2,-1)

        if col_dry is not None:
            tau = col_dry * torch.pow(self.ystd*tau + self.ymean,8)
            pfrac = torch.square(pfrac)
        #    ! Postprocess absorption output: reverse standard scaling and square root scaling
        #    tau(igpt,ilay,icol) = (ystd(igpt) * outp_both(igpt,ilay,icol) + ymeans(igpt))**8
        #    ! Optical depth from cross-sections
        #    tau(igpt,ilay,icol) = tau(igpt,ilay,icol)*col_dry_wk(ilay,icol)
        #    ! Planck fraction: reverse square root scaling
        #    pfrac(igpt,ilay,icol) = outp_both(igpt+ngpt,ilay,icol)*outp_both(igpt+ngpt,ilay,icol)
        #y(igpt,j) = ystd(igpt)*y(igpt,j) + ymean(igpt); y(igpt,j) = y(igpt,j)**8; y(igpt,j) = y(igpt,j) * layer_dry_air_molecules(j)
    
        return tau, pfrac

        
     


In [8]:
cuda = torch.cuda.is_available() 
device = torch.device("cuda" if cuda else "cpu")
print(device)


cuda


In [9]:
from torchinfo import summary

ng = 32
nn_lw = gasopt_mlp_lw(ng, xnorm_lw_min, xnorm_lw_max, 
                    ynorm_lw_mean, ynorm_lw_std,
                    nn_lw_w1, nn_lw_w2, nn_lw_w3,
                    nn_lw_b1, nn_lw_b2, nn_lw_b3)

infostr = summary(nn_lw)
print(infostr)

gasopt_mlp_lw number of original g-points: 128, new: 32, hidden neurons: 64, add MLP: True
Layer (type:depth-idx)                   Param #
gasopt_mlp_lw                            --
├─Softsign: 1-1                          --
├─Linear: 1-2                            (1,216)
├─Linear: 1-3                            (4,160)
├─Linear: 1-4                            (16,640)
├─Linear: 1-5                            16,448
Total params: 38,464
Trainable params: 16,448
Non-trainable params: 22,016


In [127]:
nn_lw.mlp1.weight.requires_grad

False

In [121]:
nn_lw = nn_lw.to("cpu")


In [117]:
nn_lw.mlp2.weight.device

device(type='cuda', index=0)

In [118]:
nn_lw.xmin.device

device(type='cuda', index=0)

In [119]:
nn_lw.mlp1.bias

Parameter containing:
tensor([-1.2667,  1.2656,  1.1475,  0.4255,  0.3181,  2.2711,  0.2568,  0.8041,
        -0.5311, -1.3564,  2.3590,  1.2974, -1.3866,  0.0314, -0.5104,  0.3856,
        -0.7240,  0.0497, -0.6619,  0.4323,  0.4457, -1.0575, -0.6047, -0.3398,
         1.1693, -0.7557, -0.4013,  0.9628,  1.4382, -0.5863,  0.4851,  1.8005,
        -0.0195,  0.2152,  0.6085, -0.2133, -0.0336,  1.5539, -0.1730,  0.2941,
        -0.1324,  1.3663, -2.4562,  0.8261, -0.1545, -0.1461, -3.1359,  0.8831,
         0.8401,  1.7644,  2.1952,  0.0637,  1.4564, -1.1052, -0.2221,  1.4904,
        -0.3267, -1.6242,  0.7764,  0.2285, -1.2039, -0.0990, -0.2626, -0.7898],
       device='cuda:0', requires_grad=True)

In [152]:
x = torch.zeros((512, 18))

tau, pfrac = nn_lw(x)

print(tau.shape, tau.min(), tau.max())
print(pfrac.shape, pfrac.min(), pfrac.max())

torch.Size([512, 32]) tensor(-1.8844, grad_fn=<MinBackward1>) tensor(2.2823, grad_fn=<MaxBackward1>)
torch.Size([512, 32]) tensor(-2.0674, grad_fn=<MinBackward1>) tensor(2.1195, grad_fn=<MaxBackward1>)


In [15]:
sw_file1 = data_path + 'rrtmgp-data-sw-g112-210809_NN_GCM_NWP_absorption.nc'
sw_file2 = data_path + 'rrtmgp-data-sw-g112-210809_NN_GCM_NWP_rayleigh.nc'

sw_ds = xr.open_dataset(sw_file1)
activ_str = sw_ds.nn_activation
input_str = sw_ds.nn_inputs
ynorm_str = sw_ds.output_scaling_info
xnorm_str = sw_ds.input_scaling_info

print("activation funcs:", activ_str[0].values, activ_str[1].values)
print("inputs:", input_str.values)
print(xnorm_str)
print(ynorm_str)

activation funcs: softsign softsign
inputs: ['tlay' 'play' 'h2o' 'o3' 'co2' 'n2o' 'ch4']
To get the required NN inputs, do the following: x(i) = log(x(i)) for i=pressure; x(i) = x(i)**(1/4) for i=H2O and O3; x(i) = (x(i) - xmin(i)) / (xmax(i) - xmin(i)) for all inputs
Model predicts scaled cross-sections. Given the raw NN output y, do the following to obtain optical depth: y(igpt,j) = ystd(igpt)*y(igpt,j) + ymean(igpt); y(igpt,j) = y(igpt,j)**8; y(igpt,j) = y(igpt,j) * layer_dry_air_molecules(j)


In [13]:
sw_ds2 = xr.open_dataset(sw_file2)

activ_str = sw_ds2.nn_activation
input_str = sw_ds2.nn_inputs
ynorm_str = sw_ds2.output_scaling_info
xnorm_str = sw_ds2.input_scaling_info

print("activation funcs:", activ_str[0].values, activ_str[1].values)
print("inputs:", input_str.values)
print(xnorm_str)
print(ynorm_str)

# lw inputs: ['tlay' 'play' 'h2o' 'o3' 'co2' 'ch4' 'n2o' 'cfc11' 'cfc12' 'co' 'ccl4'
# 'cfc22' 'hfc143a' 'hfc125' 'hfc23' 'hfc32' 'hfc134a' 'cf4']


activation funcs: softsign softsign
inputs: ['tlay' 'play' 'h2o' 'o3' 'co2' 'n2o' 'ch4']
To get the required NN inputs, do the following: x(i) = log(x(i)) for i=pressure; x(i) = x(i)**(1/4) for i=H2O and O3; x(i) = (x(i) - xmin(i)) / (xmax(i) - xmin(i)) for all inputs
Model predicts scaled cross-sections. Given the raw NN output y, do the following to obtain optical depth: y(igpt,j) = ystd(igpt)*y(igpt,j) + ymean(igpt); y(igpt,j) = y(igpt,j)**8; y(igpt,j) = y(igpt,j) * layer_dry_air_molecules(j)


In [18]:
nn_sw_w1 = sw_ds['nn_weights_1'][:].values
nn_sw_w2 = sw_ds['nn_weights_2'][:].values
nn_sw_w3 = sw_ds['nn_weights_3'][:].values

nn_sw_b1 = sw_ds['nn_bias_1'][:].values
nn_sw_b2 = sw_ds['nn_bias_2'][:].values
nn_sw_b3 = sw_ds['nn_bias_3'][:].values

ynorm_sw_mean = sw_ds['nn_output_coeffs_mean'][:].values 
ynorm_sw_std =  sw_ds['nn_output_coeffs_std'][:].values 

xnorm_sw_max = sw_ds['nn_input_coeffs_max'][:].values 
xnorm_sw_min =  sw_ds['nn_input_coeffs_min'][:].values 

print(xnorm_sw_max.shape[0])
print(ynorm_sw_std.shape[0])


7
112
